[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [APIs and JSON](https://johnfisher-ai.github.io/Python-Visual-Guides/apis-and-json.html)

# Headers and Content Types &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The cell below rebuilds what the notebook set up, including the practice API and `media_type`. Run
it first.


In [1]:
import importlib
import sys
import urllib.request
from pathlib import Path

import requests

PRACTICE_API = "https://raw.githubusercontent.com/johnfisher-ai/Python-Visual-Guides/main/notebooks/apis-and-json/practice_api.py"

if "google.colab" in sys.modules or not Path("practice_api.py").exists():
    urllib.request.urlretrieve(PRACTICE_API, "practice_api.py")    # in Colab, on every run

import practice_api
importlib.reload(practice_api)    # runs the file as it is now, not a copy imported earlier

BASE = practice_api.start()


def media_type(content_type):
    """Split a Content-Type into its media type, lowercased, and a dictionary of its parameters."""
    kind, *parameters = content_type.split(";")
    pairs = (parameter.strip().split("=", 1) for parameter in parameters if "=" in parameter)
    return kind.strip().lower(), {name.lower(): value.strip('"') for name, value in pairs}


print("ready:", BASE)


ready: http://127.0.0.1:8765


**1.** A header of your own, as the server received it.


In [2]:
received = requests.get(f"{BASE}/echo/headers", headers={"Accept-Language": "nb, en;q=0.8"}, timeout=10).json()["headers"]

print(received["Accept-Language"])


nb, en;q=0.8


`Accept-Language` asks for a language the way `Accept` asks for a format, with the same `q`
preferences: Norwegian Bokmål first, then English.


**2.** Every header of a response.


In [3]:
response = requests.get(f"{BASE}/network/export", timeout=10)

for name, value in response.headers.items():
    print(f"{name}: {value}")


Server: PracticeAPI/1.0
Date: Sun, 01 Mar 2026 09:00:00 GMT
Content-Type: application/x-ndjson
Content-Length: 1427


`items` gives each header's name and value, the names as the server wrote them. `Content-Type` says
that the body is JSON Lines.


**3.** The table as HTML.


In [4]:
response = requests.get(f"{BASE}/network/summary", headers={"Accept": "text/html"}, timeout=10)

print(response.headers["Content-Type"])
for line in response.text.splitlines()[:3]:
    print(line)


text/html; charset=utf-8
<!doctype html>
<html>
<head><title>Stations</title></head>


The same table, as a page a browser would show, with `charset=utf-8` so that `Tromsø` displays
correctly.


**4.** A parameter that is there, and one that is not.


In [5]:
for label, url in [("home page", BASE), ("station", f"{BASE}/stations/tromso")]:
    _, parameters = media_type(requests.get(url, timeout=10).headers["Content-Type"])
    print(label, parameters.get("charset"))


home page utf-8
station None


`get` on the dictionary of parameters returns `None` for the station, whose `Content-Type` is plain
`application/json`. requests decodes its body as UTF-8 anyway, because JSON must be UTF-8.


**5.** An ETag for each format.


In [6]:
url = f"{BASE}/network/summary"
json_etag = requests.get(url, headers={"Accept": "application/json"}, timeout=10).headers["ETag"]
csv_etag = requests.get(url, headers={"Accept": "text/csv"}, timeout=10).headers["ETag"]

print(json_etag, csv_etag)
print(requests.get(url, headers={"Accept": "application/json", "If-None-Match": csv_etag}, timeout=10).status_code)


"18f2fbb7de5a6cd0" "2b9fa8272109b173"
200


The CSV's `ETag` names a version of the CSV, not of the JSON, so the request for JSON got the whole
body, with a `200`. That is why the notebook's `Client` keeps each body under its path and its
format together.


**6.** The format a server chose.


In [7]:
def chosen(accept):
    """The media type /network/summary sends for an Accept header, or None for a 406."""
    response = requests.get(f"{BASE}/network/summary", headers={"Accept": accept}, timeout=10)
    if response.status_code == 406:
        return None
    return media_type(response.headers["Content-Type"])[0]


for accept in ["text/plain", "text/*", "*/*;q=0.1, text/html;q=0.2"]:
    print(f"{accept:<27} {chosen(accept)}")


text/plain                  None
text/*                      text/csv
*/*;q=0.1, text/html;q=0.2  text/html


The practice API has no plain text. `text/*` matches CSV and HTML equally, so it sends the one it
lists first. In the last header every format is acceptable, and HTML carries the highest quality.


---

&#8592; **Back to:** [Headers and Content Types](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/apis-and-json/08-headers-and-content-types.ipynb)  &nbsp;&middot;&nbsp;  [APIs and JSON Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/apis-and-json.html)
